In [ ]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

In [ ]:
import os
from pathlib import Path
import numpy as np
from openpyxl import load_workbook

In [ ]:
root = "./var/WAT"

In [ ]:
def list_files(directory: str, suffix: str) -> list[Path]:
    """Recursively list only files in `directory` with the given `suffix`."""
    return [p for p in Path(directory).rglob(f"*{suffix}") if p.is_file()]

In [ ]:
archives = list_files(root, "xlsx")
archives

In [ ]:
def find_table_header_row(ws):
    for row in ws.rows:
        for c in row:
            if c.value == "TOTALS:":
                return ws[c.row + 1]
    return None

In [ ]:
def find_formula_row_index(ws):
    for row in ws.rows:
        for c in row:
            if c.value == "TOTALS:":
                return c.row
    return None

In [ ]:
def max_content_col(row):
    for cell in reversed(row):
        if cell.value:
            return cell.column_letter
    return row[0].column_letter

In [ ]:
def partition_sheet(ws):
    result = {}
    f_row = find_formula_row_index(ws)
    if f_row is not None:
        h_row = f_row + 1
        result["table_header"] = f"A{h_row}:{max_content_col(ws[h_row])}{h_row}"
        max_col = "A"
        for row in range(1, f_row-1):
            max_col = max(max_col, max_content_col(ws[row]))
        result["sheet_header"] = f"A1:{max_col}{f_row-1}"
        max_col = "A"
        for row in range((h_row+1), ws.max_row):
            max_col = max(max_col, max_content_col(ws[row]))
            if not ws[row][0].value:
                result["table_content"] = f"A{h_row+1}:{max_col}{row-1}"
                break
    return result

In [ ]:
def extract_strings_in_range(ws, cell_range):
    result = dict()
    for row in ws[cell_range]:
        for cell in row:
            value = cell.value
            if value is not None:
                if isinstance(value, float):
                    value = int(value)
                if not isinstance(value, str):
                    value = str(value)
                if value:
                    result[value] = result.get(value, 0) + 1
    return result

In [ ]:
def extract_strings_by_partition(ws):
    return { key: extract_strings_in_range(ws, cell_range) 
             for key, cell_range in partition_sheet(ws).items() }

In [ ]:
def all_keys(d1, d2):
    return set(d1.keys()).union(d2.keys())

In [ ]:
def merge_hist(hist1, hist2):
    return { key: hist1.get(key, 0) + hist2.get(key, 0)
        for key in all_keys(hist1, hist2) }

In [ ]:
def ranked_hist(hist):
    return sorted(list(hist.items()), key=lambda x: -x[1])

In [ ]:
def truncate_hist(hist, thresh=0, percentile=None):
    hist = ranked_hist(hist)
    if percentile is not None:
        thresh = sum([x[1] for x in hist]) * percentile
    return list(filter(lambda x: x[1] >= thresh, hist))

In [ ]:
def merge_all_hists(a, b):
    return { 
        key: merge_hist(a.get(key, dict()), b.get(key, dict())) 
        for key in all_keys(a, b) }

In [ ]:
def reduce_archive(archives, map_fn, reduce_fn, init=None):
    result = init
    for i, archive in enumerate(archives):
        print(i, archive)
        wb = load_workbook(archive)
        result = reduce_fn(map_fn(wb), result)
    return result    

In [ ]:
archive_strings = reduce_archive(archives,
                         lambda a: extract_strings_by_partition(a.worksheets[0]),
                         lambda a, b: { 
                            key: merge_hist(a.get(key, {}), b.get(key, {})) for key in all_keys(a, b) 
                            },
                         dict())

In [ ]:
ranked_hist(strings["archive_strings"])

In [ ]:
def archive_mapper(archive):
    result = { "archive": extract_strings_by_partition(archive.worksheets[0]) }
    for sheet in archive.worksheets[1:]:
        strings = extract_strings_by_partition(sheet)
        subset = "fund" if sheet.title.startswith("fund") else "opus"
        result[subset] = merge_all_hists(strings, result.get(subset, dict()))
    return result

def archive_reducer(a, b):
    return { 
        key: merge_all_hists(a.get(key, dict()), b.get(key, dict()))
        for key in all_keys(a, b) }

In [ ]:
result = reduce_archive(archives, archive_mapper, archive_reducer, dict())

In [ ]:
truncate_hist(result['fund']['table_header'], 10)

In [ ]:
truncate_hist(result['opus']['sheet_header'], 200)

In [ ]:
truncate_hist(result['archive']['table_content'], percentile=.05)

In [ ]:
for subset in ['archive', 'fund', 'opus']:
    hists = result[subset]
    for key in hists.keys():
        print(f"======= {subset}:{key}: ========")
        print(truncate_hist(result[subset][key], percentile=.05))